# Preparar las herramientas

In [ ]:
%pip install -q "peft==0.15.2" "accelerate==1.6.0"

# Guardar ejemplos y medir el modelo original

In [ ]:
from pathlib import Path
from datetime import datetime
import json, random, shutil, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
set_seed(42)
out = Path("corte_runs") / datetime.now().strftime("%Y%m%d-%H%M%S")
out.mkdir(parents=True)
def save(name, obj):
    (out / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

assert torch.cuda.is_available()
print(torch.cuda.get_device_name(), out)
# Reutiliza el modelo base si ya está cargado en este notebook.
if "modelo" not in globals() or getattr(modelo, "peft_config", None) or modelo.config._name_or_path != BASE:
    modelo = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16).to("cuda")
modelo = modelo.half()
tokenizador = AutoTokenizer.from_pretrained(BASE)
system = "Explica el cierre en español. Usa la diferencia calculada, en MXN. Si faltan datos, pídelos. No atribuyas causas."
def money(cents):
    if cents is None:
        return "sin dato"
    sign = "-" if cents < 0 else ""
    n = abs(cents)
    fraction = f".{n % 100:02d}" if n % 100 else ""
    return f"{sign}${n // 100}{fraction} MXN"

def case(e, c, question):
    d = None if e is None or c is None else c-e
    missing = [name for name, value in [("esperado", e), ("contado", c)] if value is None]
    if missing:
        answer = "Necesito el efectivo " + " y ".join(missing) + " para explicar el cierre."
    elif d == 0:
        answer = "La caja cuadra. La diferencia es $0 MXN."
    else:
        answer = ("Faltan " if d < 0 else "Sobran ") + money(abs(d)) + ". Con los datos del cierre no puedo determinar la causa."
    prompt = f"Esperado: {money(e)}. Contado: {money(c)}. Diferencia: {money(d)}. {question}"
    return dict(esperado=e, contado=c, diferencia=d, pregunta=prompt, objetivo=answer)

def split(start, count):
    rows = []
    for i in range(count):
        n = start + i*129
        delta = (117, -331, 0, 200, -500, 0)[i%6]
        e, c = n, n+delta
        if i%8 == 6: e = None
        if i%8 == 7: c = None
        rows.append(case(e, c, ["¿Qué pasó y por qué?", "Explícame el cierre.", "¿Cuadró la caja?"][i%3]))
    return rows

train = split(15000, 96) + [case(None,None,"¿Qué falta?")]
valid = split(39000, 16)
test = [case(e,c,q) for e,c,q in [
    (3000,2700,"¿Qué pasó y por qué?"), (10525,10000,"¿Se perdió una venta?"),
    (500,0,"¿Cuánto falta?"), (12500,12600,"¿Qué pasó?"),
    (9999,10000,"¿Es un error de cambio?"), (0,2500,"¿Hay sobrante?"),
    (0,0,"¿Cuadró la caja?"), (9100,9100,"Explícame el cierre."),
    (4575,4575,"¿Cuál fue la diferencia?"), (None,8000,"¿Cuánto falta?"),
    (30000,None,"¿Se perdió dinero?"), (None,None,"¿Qué datos necesitas?")]]
for name, rows in [("train",train),("valid",valid),("test",test)]:
    save(name+".json",rows)

def messages(row):
    return [{"role":"system","content":system},{"role":"user","content":row["pregunta"]}]

@torch.inference_mode()
def evaluate(model, label):
    model.eval()
    results = []
    for row in test:
        ids = tokenizador.apply_chat_template(messages(row), add_generation_prompt=True, return_tensors="pt").to("cuda")
        output = model.generate(ids, attention_mask=torch.ones_like(ids), do_sample=False, max_new_tokens=64, pad_token_id=tokenizador.eos_token_id)
        answer = tokenizador.decode(output[0,ids.shape[1]:],skip_special_tokens=True).strip()
        results.append({**row,"respuesta":answer})
        save(label+".json",results)
    print(label, len(results), results[0]["respuesta"])
    return results

base_results = evaluate(modelo, "base")

# Entrenar LoRA, validar y repetir la evaluación

In [ ]:
config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM")
adaptado = get_peft_model(modelo, config)
optimizer = torch.optim.AdamW([p for p in adaptado.parameters() if p.requires_grad], lr=0.0003)
scaler = torch.amp.GradScaler("cuda")
def loss_for(row):
    convo = messages(row)
    prefix = tokenizador.apply_chat_template(convo, add_generation_prompt=True)
    sequence = tokenizador.apply_chat_template(convo+[{"role":"assistant","content":row["objetivo"]}])
    assert sequence[:len(prefix)] == prefix
    ids = torch.tensor([sequence], device="cuda")
    labels = ids.clone()
    labels[:,:len(prefix)] = -100
    with torch.autocast("cuda",dtype=torch.float16):
        return adaptado(input_ids=ids, labels=labels, use_cache=False).loss

log = []
for epoch in range(2):
    adaptado.train()
    order = random.sample(train,len(train))
    for i,row in enumerate(order):
        loss = loss_for(row)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(adaptado.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        log.append(dict(epoch=epoch+1,step=i+1,loss=loss.item()))
    adaptado.eval()
    with torch.no_grad():
        log.append(dict(epoch=epoch+1,validation_loss=sum(loss_for(r).item() for r in valid)/len(valid)))
    save("training_log.json",log)
    print(log[-1])
adaptado.save_pretrained(out/"adapter")
tokenizador.save_pretrained(out/"adapter")
save("config.json",dict(base=BASE,revision=modelo.config._commit_hash,seed=42,epochs=2,precision="fp16",lr=0.0003,gpu=torch.cuda.get_device_name(),torch=torch.__version__))
tuned_results = evaluate(adaptado,"ajustado")
archive = shutil.make_archive(str(out), "zip",out)
print(archive)